In [1]:
%%script false --no-raise-error
!pip3 install opencv-python numpy matplotlib opendatasets pandas kagglehub

In [2]:
%%script false --no-raise-error
!kaggle datasets download -d adamelkholy/human-ai-artwork
!mkdir data
!unzip human-ai-artwork.zip -d data
!rm *.zip

In [3]:
## Package Imports

In [4]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Hide INFO & WARNING logs

import tensorflow as tf
tf.get_logger().setLevel("ERROR")  # Only show errors



import time
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np 


## Pre-Processing

In [5]:
### Splits and Tuning

In [6]:
weights = {0: 1.0, 1: 1.0} 
num_classes = 1
batch_size = 32
img_ht = 256
img_wt = 256
data_dir = "./data/data"

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.4,
    subset="training",
    seed=104,
    image_size=(img_ht, img_wt),
    batch_size = batch_size
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.4,
    subset="validation",
    seed=104,
    image_size=(img_ht, img_wt),
    batch_size=batch_size
)

val_batches = int(0.5 * len(val_test_ds))
val_ds = val_test_ds.take(val_batches)
test_ds = val_test_ds.skip(val_batches)

"""
def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Scale pixel values to [0,1]
    return image, label
""" 



Found 271993 files belonging to 52 classes.
Using 163196 files for training.


I0000 00:00:1739223841.845119  325974 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739223841.845355  325974 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739223841.874432  325974 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739223841.874643  325974 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

Found 271993 files belonging to 52 classes.
Using 108797 files for validation.


'\ndef normalize(image, label):\n    image = tf.cast(image, tf.float32) / 255.0  # Scale pixel values to [0,1]\n    return image, label\n'

### Imbalance Bias

In [7]:
pos = 190549
neg = 81457
in_bias = np.log([pos/neg])
out_bias = tf.keras.initializers.Constant(in_bias)

### Image to Binary Mappings

In [9]:
def image_to_binary(img, label):
    label = tf.cast(label, tf.int32)
    new_label = tf.where(label < 25, 1, 0) # label is number of ai class folders
    new_label = tf.expand_dims(new_label, axis=-1)
    return (img, new_label)


train_ds = train_ds.map(image_to_binary)
val_ds = val_ds.map(image_to_binary)
test_ds = test_ds.map(image_to_binary)

In [ ]:
## Data Augmentation

In [10]:
def img_augmentation(img, lbl):
    min_scale = 0.5
    max_scale = 2.0
    scale_factor = tf.random.uniform(shape=[], minval=min_scale, maxval=max_scale)
    resized_img = tf.image.resize(img, tf.cast(tf.cast(tf.shape(img)[1:3], tf.float32) * scale_factor, tf.int32))
    rescaled_img = tf.image.resize(resized_img, tf.shape(img)[1:3])
    
    return (rescaled_img, lbl)

train_ds = train_ds.map(img_augmentation)
val_ds = val_ds.map(img_augmentation)
test_ds = test_ds.map(img_augmentation)
"""
def reshape(image, label):
    label = tf.reshape(label, (-1, 1))  # Convert (batch_size,) to (batch_size, 1)
    return image, label

train_ds = train_ds.map(reshape)
val_ds = val_ds.map(reshape)
test_ds = test_ds.map(reshape)
"""

'\ndef reshape(image, label):\n    label = tf.reshape(label, (-1, 1))  # Convert (batch_size,) to (batch_size, 1)\n    return image, label\n\ntrain_ds = train_ds.map(reshape)\nval_ds = val_ds.map(reshape)\ntest_ds = test_ds.map(reshape)\n'

## Model Training

In [11]:
dejaivu_model = tf.keras.Sequential([
  tf.keras.layers.Rescaling(1./255),

  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.MaxPooling2D(),

  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.MaxPooling2D(),

  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_classes, bias_initializer=out_bias, activation="sigmoid")
])

dejaivu_model.name = "dejAIvu_model"
model_path = "./"

### Metrics

In [12]:
metrics = [
      tf.keras.metrics.BinaryAccuracy(name='accuracy'),
      tf.keras.metrics.BinaryCrossentropy(name='cross entropy'), # equiv. to model's loss
      tf.keras.metrics.MeanSquaredError(name='MSE'),
      tf.keras.metrics.TruePositives(name='tp'),
      tf.keras.metrics.FalsePositives(name='fp'),
      tf.keras.metrics.TrueNegatives(name='tn'),
      tf.keras.metrics.FalseNegatives(name='fn'),
      tf.keras.metrics.Precision(name='precision'),
      tf.keras.metrics.Recall(name='recall'),
      tf.keras.metrics.AUC(name='roc', curve='ROC'),             # receiver operating characteristic curve
      tf.keras.metrics.AUC(name='prc', curve='PR'),              # precision-recall curve
]

class CustomHistory(tf.keras.callbacks.Callback):
    def __init__(self):
      super(CustomHistory, self).__init__()
      self.losses = []
      self.prcs = []
      self.recalls = []
      self.precisions = []
      self.accuracies = []
      self.mses = []

    """ called upon completion of each batch during training, records all performance metrics """
    def on_train_batch_end(self, batch, logs=None):
      self.losses.append(logs['loss'])
      self.mses.append(logs['MSE'])
      self.accuracies.append(logs['accuracy'])
      self.prcs.append(logs['prc'])
      self.recalls.append(logs['recall'])
      self.precisions.append(logs['precision'])

    """ called upon completion of each batch during testing, records all performance metrics """
    def on_test_batch_end(self, batch, logs=None):
      self.losses.append(logs['loss'])
      self.mses.append(logs['MSE'])
      self.accuracies.append(logs['accuracy'])
      self.prcs.append(logs['prc'])
      self.recalls.append(logs['recall'])
      self.precisions.append(logs['precision'])

    """ return all performance metrics """
    def get_metrics(self):
      return self.losses, self.mses, self.accuracies, self.prcs, self.recalls, self.precision

## Model Training

### Compiling and Evaluating Helpers

In [20]:
def compile_model(model):
  model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=metrics
  )
  return model

def fit_model(model):
  history_callback = CustomHistory()
  model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    class_weight=weights,
    callbacks=[history_callback]
  )
  return model, history_callback

def evaluate_model_on_test(model):
  eval_metrics = model.evaluate(test_ds)
  return eval_metrics

def save_model(model):
  model_name = model.name
  print("\nSaving " + model_name + ".keras")
  try:
    model.save(model_path + model_name + ".keras")
  except:
    print("Error saving " + model_name + ".keras...")
    return
  print(model_name + ".keras saved successfully.\n")
  
def save_data(data, filename):
  print("Saving data for " + filename)
  try:
    with open(model_path + filename+".txt", 'w') as writefile:
      writefile.write(str(data))
  except:
    print("Error saving data for " + filename)
    return
  print("Data saved.\n")

### Execute Training


In [ ]:
%%script false --no-raise-error
from tensorflow.python.client import device_lib
from tensorflow.keras import mixed_precision

policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 1749858783208598932
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 48935665664
locality {
  bus_id: 1
  links {
    link {
      device_id: 1
      type: "StreamExecutor"
      strength: 1
    }
  }
}
incarnation: 12801636815138379687
physical_device_desc: "device: 0, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
, name: "/device:GPU:1"
device_type: "GPU"
memory_limit: 48152903680
locality {
  bus_id: 1
  links {
    link {
      type: "StreamExecutor"
      strength: 1
    }
  }
}
incarnation: 8912503634688768045
physical_device_desc: "device: 1, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:41:00.0, compute capability: 8.9"
xla_global_id: 2144165316
]


I0000 00:00:1739219780.006796  323221 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739219780.007100  323221 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739219780.007294  323221 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1739219780.007469  323221 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [ ]:
%%script false --no-raise-error
for x_batch, y_batch in train_ds.take(1):
    print("Sample input (X):", x_batch.numpy())
    print("Sample labels (Y):", y_batch.numpy())
    break


Sample input (X): [[[[0.4861358  0.35127726 0.17382343]
   [0.58210784 0.4472493  0.2696952 ]
   [0.49695638 0.36219603 0.18426642]
   ...
   [0.7278149  0.6363772  0.39449307]
   [0.93659824 0.9154526  0.80366004]
   [0.98507065 0.9950761  0.9814844 ]]

  [[0.48545903 0.34606963 0.22543305]
   [0.4898536  0.35135382 0.22567187]
   [0.44865152 0.3113598  0.17641076]
   ...
   [0.74818397 0.65057784 0.41073957]
   [0.9411489  0.91697264 0.8021397 ]
   [0.9853619  0.99488926 0.9792806 ]]

  [[0.6098067  0.48824254 0.36193448]
   [0.59115577 0.47495446 0.32167104]
   [0.5183793  0.40635177 0.22330552]
   ...
   [0.74761915 0.6186189  0.389869  ]
   [0.91110826 0.8658689  0.7466707 ]
   [0.98997873 0.9867298  0.9722339 ]]

  ...

  [[0.5324994  0.46953976 0.408404  ]
   [0.5384369  0.46436897 0.4043642 ]
   [0.3465269  0.26415202 0.20181392]
   ...
   [0.7847055  0.7042545  0.6622349 ]
   [0.7553624  0.7046421  0.61723757]
   [0.8066545  0.7495775  0.64823645]]

  [[0.49780214 0.43596426 0

In [22]:
from tensorflow.keras.metrics import MeanSquaredError, AUC, Recall, Precision
"""
def train_model_pipeline(lr=0.001):
    start = time.time()

    # Enable Multi-GPU Training
    strategy = tf.distribute.MirroredStrategy()
    print(f"Using {strategy.num_replicas_in_sync} GPU(s) for training.")

    # Define model inside `strategy.scope()` to avoid errors
    with strategy.scope():
        dejaivu_model = tf.keras.Sequential([
            tf.keras.layers.Rescaling(1./255),

            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.MaxPooling2D(),

            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.Conv2D(32, 3, activation='relu'),
            tf.keras.layers.MaxPooling2D(),

            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dense(num_classes, bias_initializer=out_bias, activation="sigmoid")
        ])
        
        dejaivu_model.name = "dejAIvu_model"

        # Compile the model inside the scope
        dejaivu_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0), 
                              loss="binary_crossentropy", 
                              metrics=["accuracy", Precision(name="precision"), Recall(name="recall"), AUC(name="prc", curve="PR"), MeanSquaredError(name="MSE")])
    


    label_counts = np.array([0, 0])  # Assuming binary labels [0,1]
    for img, lbl in train_ds.unbatch().take(1000):  # Check 1000 samples
        label_counts[int(lbl.numpy())] += 1

    print(f"Class 0: {label_counts[0]} samples, Class 1: {label_counts[1]} samples")
    
    tf.config.run_functions_eagerly(False)

    
    # Train the model inside multi-GPU strategy
    trained_model, history = fit_model(dejaivu_model)

    # Save model and history
    save_data(history.get_metrics(), dejaivu_model.name + "_history")
    save_model(trained_model)

    # Evaluate on test set
    evals = evaluate_model_on_test(trained_model)
    save_data(evals, dejaivu_model.name + "_evals")

    time_taken = time.time() - start
    print(f"Training complete in {round(time_taken/60, 2)} minutes")

    return trained_model, evals, history
"""

def train_model_pipeline(model, lr=0.001):
  start = time.time()
  model_name = model.name
  print("Now training " + model_name)

  # compile model and fit to training data
  compiled_model = compile_model(model)
  compiled_model.optimizer.learning_rate = lr
  trained_model, history = fit_model(compiled_model)

  # save model.keras and history data
  save_data(history.get_metrics(), model_name+"_history")
  save_model(trained_model)

  # evaluate on test set and save evaluation
  evals = evaluate_model_on_test(trained_model)
  save_data(evals, model_name + "_evals")

  time_taken = time.time() - start
  print("Training complete in", round((time_taken)/60, 2), "minutes")
  return trained_model, evals, history


In [17]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [23]:
# Run training with your custom model inside the correct scope
model, evals, history = train_model_pipeline(dejaivu_model)

Now training dejAIvu_model
Epoch 1/3


I0000 00:00:1739223997.141425  326337 service.cc:146] XLA service 0x7f3f240062e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1739223997.141454  326337 service.cc:154]   StreamExecutor device (0): NVIDIA RTX 6000 Ada Generation, Compute Capability 8.9
I0000 00:00:1739223997.141458  326337 service.cc:154]   StreamExecutor device (1): NVIDIA RTX 6000 Ada Generation, Compute Capability 8.9


   5/5100 ━━━━━━━━━━━━━━━━━━━━ 2:53 34ms/step - MSE: 0.3242 - accuracy: 0.5971 - cross entropy: 1.0985 - fn: 16.0000 - fp: 22.4000 - loss: 1.0985 - prc: 0.6521 - precision: 0.6734 - recall: 0.7557 - roc: 0.4771 - tn: 9.6000 - tp: 48.0000        

I0000 00:00:1739224003.993592  326337 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2825/5100 ━━━━━━━━━━━━━━━━━━━━ 1:12 32ms/step - MSE: 0.1360 - accuracy: 0.8051 - cross entropy: 0.4265 - fn: 2201.3960 - fp: 5827.4429 - loss: 0.4265 - prc: 0.9174 - precision: 0.8175 - recall: 0.9333 - roc: 0.8422 - tn: 7661.2764 - tp: 29525.8848

KeyboardInterrupt: 